### Prompt EngineeringThe prompt is the interface, it shapes the distribution the model conditions on, and can do a real amount of the work fine-tuning would otherwise have to do (`pretraining-and-finetuning.ipynb`). Covers zero-shot vs. few-shot, chain-of-thought, self-consistency, structured-output prompting, system/role prompts, and why prompts are brittle enough that this matters for eval (`llm-evals.ipynb`).

#### 0. Setup: instruct model, not basePrompting technique (as opposed to raw next-token continuation) only makes sense on an instruction-tuned model, a base model (see `pretraining-and-finetuning.ipynb`'s base vs. instruct comparison) tends to CONTINUE a prompt rather than follow it as an instruction, so every demo below uses Qwen2.5-0.5B-Instruct.

In [ ]:
import torchfrom transformers import AutoTokenizer, AutoModelForCausalLMmodel_name = "Qwen/Qwen2.5-0.5B-Instruct"device = "cuda" if torch.cuda.is_available() else "cpu"dtype = torch.float16 if device == "cuda" else torch.float32tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype, trust_remote_code=True).to(device)model.eval()def ask(user_message, system_message=None, max_new_tokens=80):    messages = ([{"role": "system", "content": system_message}] if system_message else [])    messages.append({"role": "user", "content": user_message})    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)    inputs = tokenizer(prompt, return_tensors="pt").to(device)    with torch.no_grad():        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

#### 1. Zero-shot vs. few-shotZero-shot: just the instruction, no examples, the model has to infer the desired output format purely from the instruction's wording. Few-shot: a handful of (input, output) example pairs included in the prompt itself, narrowing both WHAT to answer and, just as importantly, the exact FORMAT to answer in, no separate fine-tuning run needed, the examples live in the prompt.Worked comparison, the same fraud-typology classification task `fraud-theme-detection`'s own feature extraction does: zero-shot only states the task, few-shot adds 2 labeled examples first, forcing the exact output format (a single lowercase label, no punctuation, no restating the narrative) the way a naive zero-shot prompt often fails to.

In [ ]:
zero_shot_prompt = (    "Classify this complaint narrative into a fraud typology: "    '"Someone called claiming to be from the IRS, demanded payment via gift cards immediately."')few_shot_prompt = (    "Classify each complaint narrative into a fraud typology. Answer with a single lowercase "    "label only.\n\n"    'Narrative: "A man posing as my grandson called saying he was arrested and needed bail money."\n'    "Label: family_emergency_scam\n\n"    'Narrative: "I received a text saying my bank account was locked, click this link to verify."\n'    "Label: phishing_scam\n\n"    'Narrative: "Someone called claiming to be from the IRS, demanded payment via gift cards immediately."\n'    "Label:")print("ZERO-SHOT:\n", ask(zero_shot_prompt))print("\nFEW-SHOT:\n", ask(few_shot_prompt))

#### 2. Chain-of-thought (CoT) promptingAppending "let's think step by step" (or providing worked reasoning examples) makes the model generate intermediate reasoning tokens BEFORE the final answer, instead of jumping straight to one. Why it helps: each generated token only gets one forward pass' worth of computation (`llm-architecture.ipynb`), a model asked to answer a multi-step problem in a single token has nowhere to "do the work", CoT gives it that space, the intermediate tokens ARE the extra computation, not just a nicer-looking explanation.Directly connects to a real bug hit earlier in this project's own work: a reasoning model (gpt-5-nano) silently burned its entire token budget on internal chain-of-thought reasoning (`completion_tokens_details.reasoning_tokens`) before ever emitting the answer, returning `content=None`. Reasoning-capable models do CoT internally by default now, an explicit "think step by step" prompt matters most for models that DON'T reason internally on their own.

In [ ]:
direct_prompt = "A store had 23 apples. They sold 15 and then received a new shipment of 8. How many apples now?"cot_prompt = direct_prompt + " Let's think step by step."print("DIRECT:\n", ask(direct_prompt, max_new_tokens=40))print("\nCHAIN-OF-THOUGHT:\n", ask(cot_prompt, max_new_tokens=120))

#### 3. Self-consistencyOne CoT sample can still land on the wrong reasoning path. Self-consistency samples MULTIPLE independent CoT completions (with sampling temperature > 0, see `decoding-strategies.ipynb`, not greedy, or every sample would be identical) for the same question, then takes a majority vote over the final answers, discarding the intermediate reasoning text itself, only the answer distribution matters. Worked toy example: 5 sampled CoT completions give final answers [16, 16, 14, 16, 16], majority vote picks 16, the correct arithmetic answer above, even though one sampled reasoning path (14) went wrong somewhere.

In [ ]:
from collections import Counter# real: sample with temperature>0, do_sample=True, repeat N times, parse each final answer.# simulated here since a 0.5B model rarely reasons reliably enough for a clean live demo.sampled_answers = [16, 16, 14, 16, 16]majority = Counter(sampled_answers).most_common(1)[0]print("sampled final answers:", sampled_answers)print(f"self-consistency answer: {majority[0]} ({majority[1]}/{len(sampled_answers)} votes)")

#### 4. Structured-output promptingConstrain the output to a specific schema (JSON with fixed keys, `additionalProperties: false`) rather than free text, exactly the mechanism `fraud-theme-detection`'s `FEATURE_SCHEMA` uses for feature extraction. Two layers doing this together: the PROMPT (instructing the exact shape expected) and the DECODING constraint (some APIs enforce valid JSON/schema conformance at the token-sampling level, not just via instruction, so the model literally cannot emit an invalid token at that position). Prompting alone (no decoding constraint) can still emit malformed JSON on rare cases, worth remembering as a real failure mode of the technique, not a guarantee, unless the API's structured-output enforcement is actually turned on.

In [ ]:
structured_prompt = (    "Extract fields from this narrative as JSON with exactly these keys: "    '"mentions_irs" (boolean), "narrative_length_category" (one of "short","medium","long").\n'    "Return ONLY the JSON object, no other text.\n\n"    'Narrative: "Someone called claiming to be from the IRS, demanded payment via gift cards immediately."')print(ask(structured_prompt, max_new_tokens=60))print("\nwithout schema enforcement at the decoding level, this can still emit malformed JSON,")print("e.g. trailing commentary after the object, exactly why FEATURE_SCHEMA used real schema enforcement")

#### 5. System/role promptsThe system message sets persistent context/behavior for the whole conversation (tone, constraints, persona), separate from the user message that carries the actual per-turn request. Same underlying mechanism, both just become part of the token sequence via the chat template (`tokenizer.apply_chat_template` above), but keeping instructions in the system slot instead of prepending them to every user message keeps them from competing with, or being accidentally overridden by, the user's own wording.

In [ ]:
system_msg = "You are a fraud analyst. Answer in exactly one word: the fraud typology name, lowercase, no punctuation."user_msg = "Someone called claiming to be from the IRS, demanded payment via gift cards immediately."print(ask(user_msg, system_message=system_msg, max_new_tokens=15))

#### 6. Prompt brittleness, and practical patternsSmall wording changes (reordering examples, rephrasing the instruction, even whitespace) can shift outputs meaningfully, the model has no notion of "equivalent phrasing", it conditions on the literal token sequence. This is exactly why `llm-evals.ipynb`'s LLM-as-judge and eval pipelines matter, a prompt that looked fine on 3 manual examples can silently regress on the full distribution after a tiny wording tweak, only a real eval set catches that.Patterns that reduce brittleness in practice: explicit delimiters around input text (so the model doesn't confuse instruction text with the data it's operating on), explicit output-format instructions ("return ONLY...", stated above), negative constraints ("do not include..."), and decomposition, breaking one large, multi-part prompt into several smaller, single-purpose calls chained together, the same idea `agents-and-tool-use.ipynb` builds on for multi-step tasks that need actions between reasoning steps, not just more text in one call.